In [1]:
"""Импорты, путь к проекту и переменные окружения.
Ноутбук лежит в `notebooks/`, а код — в корне проекта. Делаем корень рабочим
каталогом и добавляем его в `sys.path`, чтобы `import app` работал при запуске
через Jupyter/nbconvert. `load_dotenv()` подтягивает `LLM_API_KEY` из `.env`.
"""
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")


True

In [2]:
"""Golden-датасет из 10 эталонных вопросов.
Структура: 7 английских вопросов по 3 темам корпуса (linear_model / tree
/ model_evaluation), 2 русских для проверки мультиязычного retrieval'а,
1 meta-вопрос про `about.md`. Поле `ground_truth_url_keywords` нужно для
Recall@k в Ячейке 4: считаем hit, если URL retrieved-чанка содержит
ожидаемое ключевое слово.
"""
GOLDEN = [
    # --- 7 EN in-corpus вопросов ---
    {
        "question": "How does Ridge regression handle multicollinearity?",
        "ground_truth": "Ridge adds an L2 penalty alpha * sum(w_i^2) to the loss, "
                        "which shrinks correlated coefficients toward each other.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What does the alpha parameter control in Ridge?",
        "ground_truth": "Alpha controls regularization strength; larger alpha means "
                        "stronger penalty and smaller coefficients.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What is the difference between Lasso and Ridge?",
        "ground_truth": "Lasso uses L1 penalty which can zero out coefficients (feature "
                        "selection); Ridge uses L2 which shrinks but never zeroes.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What does min_samples_leaf control in a decision tree?",
        "ground_truth": "min_samples_leaf is the minimum number of samples required to be "
                        "at a leaf node; higher values prevent overfitting by limiting depth.",
        "ground_truth_url_keywords": ["tree"],
    },
    {
        "question": "When does a decision tree overfit?",
        "ground_truth": "Trees overfit when grown too deep without min_samples_leaf or "
                        "min_samples_split constraints, memorising training noise.",
        "ground_truth_url_keywords": ["tree"],
    },
    {
        "question": "What is the formula for precision?",
        "ground_truth": "precision = TP / (TP + FP). Fraction of positive predictions that "
                        "are actually positive.",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    {
        "question": "When is recall more important than precision?",
        "ground_truth": "Recall matters most when missing positives is costly: cancer "
                        "screening, fraud detection, anything where false negatives are "
                        "more harmful than false positives.",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    # --- 2 RU вопроса (тест мультиязычного retrieval) ---
    {
        "question": "Что такое L2-регуляризация?",
        "ground_truth": "L2-регуляризация добавляет к функции потерь штраф, "
                        "пропорциональный сумме квадратов коэффициентов модели. "
                        "Используется в Ridge.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "Что такое F1-мера?",
        "ground_truth": "F1 — гармоническое среднее precision и recall, "
                        "F1 = 2 * precision * recall / (precision + recall).",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    # --- 1 meta-вопрос (тест about.md) ---
    {
        "question": "Что ты умеешь?",
        "ground_truth": "Отвечаю на вопросы по трём разделам scikit-learn: линейные "
                        "модели, деревья решений, метрики качества.",
        "ground_truth_url_keywords": ["about.md", "local"],
    },
]


In [3]:
"""Прямое подключение к RAG-pipeline без HTTP-слоя.
`build_rag_chain()` поднимает retriever (Qdrant + embedder) и LCEL-цепочку
в этом же процессе. Прямой вызов даёт чистые тайминги и доступ к
`retrieved_contexts`, которые понадобятся RAGAS в Ячейке 5.
"""
from app.config import settings
from app.rag.chain import build_rag_chain

chain, retriever = build_rag_chain()


C:\Users\sonyb\anaconda3\envs\sklearn-docs-rag\Lib\site-packages\qdrant_client\qdrant_remote.py:284: UserWarning: Qdrant client version 1.19.1 is incompatible with server version 1.12.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(
C:\Users\sonyb\anaconda3\envs\sklearn-docs-rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2013.90it/s]

In [4]:
"""Recall@k для retriever'а.
Прогоняем каждый golden-вопрос через retriever, забираем URL'ы топ-k чанков
и считаем hit по keyword-совпадению. Дополнительно копим
`retrieved_contexts` — они уйдут в RAGAS на следующем шаге.
"""
def url_match(returned_urls: list[str], expected_keywords: list[str]) -> bool:
    """Помечает retrieval как hit, если хоть один URL содержит ожидаемое ключевое слово."""
    if not expected_keywords:
        return not returned_urls
    return any(
        any(kw.lower() in url.lower() for kw in expected_keywords)
        for url in returned_urls
    )

results = []
for item in GOLDEN:
    docs = retriever.invoke(item["question"])
    urls = [doc.metadata.get("source", "") for doc in docs]
    results.append({
        "question": item["question"],
        "ground_truth": item["ground_truth"],
        "retrieved_urls": urls,
        "retrieved_contexts": [doc.page_content for doc in docs],
        "retriever_hit": url_match(urls, item["ground_truth_url_keywords"]),
    })

hits = sum(1 for r in results if r["retriever_hit"])
recall_at_k = hits / len(results)
print(f"Retriever Recall@{settings.top_k}: {recall_at_k:.3f}  ({hits}/{len(results)})")


Retriever Recall@4: 1.000  (10/10)


In [5]:
"""Generation-метрики через RAGAS.
RAGAS 0.2.x требует Dataset со строго заданными именами полей
(`user_input` / `response` / `retrieved_contexts` / `reference`).
LLM и эмбеддер оборачиваем в `LangchainLLMWrapper` и
`LangchainEmbeddingsWrapper`.
"""
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import Faithfulness, ResponseRelevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_huggingface import HuggingFaceEmbeddings

from app.llm import get_llm

for r in results:
    r["response"] = chain.invoke(r["question"])

ragas_data = Dataset.from_list([
    {
        "user_input": r["question"],
        "response": r["response"],
        "retrieved_contexts": r["retrieved_contexts"],
        "reference": r["ground_truth"],
    }
    for r in results
])

llm = LangchainLLMWrapper(get_llm())
emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(
        model_name="intfloat/multilingual-e5-small",
        encode_kwargs={"normalize_embeddings": True},
    )
)

ragas_scores = evaluate(
    dataset=ragas_data,
    metrics=[
        Faithfulness(llm=llm),
        ResponseRelevancy(llm=llm, embeddings=emb),
    ],
)
print(ragas_scores)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  63%|██████▎   | 126/199 [00:00<00:00, 1259.58it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1857.90it/s]

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Exception raised in Job[1]: OutputParserException(Invalid json output: {
    "!question!": "How! does! Ridge! regression! address! the!!!!! problem! of!! multicolline!arity?",
    "!noncommittal!": 0
!}
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )


...

Exception raised in Job[9]: TimeoutError()


Exception raised in Job[13]: TimeoutError()


Exception raised in Job[7]: TimeoutError()


Evaluating: 100%|██████████| 20/20 [03:00<00:00,  9.00s/it]

{'faithfulness': 0.8542, 'answer_relevancy': 0.8618}


In [6]:
"""Cохраняем сводку метрик в JSON.
Берём mean по каждой RAGAS-метрике, добавляем retriever-Recall@k и
метаданные прогона (модель, embedder, top_k). Файл `rag_metrics.json`
цитируется в README.
"""
df = ragas_scores.to_pandas()
gen_scores = {
    "faithfulness": float(df["faithfulness"].mean()),
    "answer_relevancy": float(df["answer_relevancy"].mean()),
}
metrics = {
    "n_questions": len(GOLDEN),
    "model": settings.llm_model,
    "embedding_model": settings.embedding_model,
    "top_k": settings.top_k,
    "retriever": {
        f"recall_at_{settings.top_k}": recall_at_k,
        "hits": hits,
    },
    "generation": gen_scores,
}
Path("notebooks/rag_metrics.json").write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False)
)
print(json.dumps(metrics, indent=2, ensure_ascii=False))


{
  "n_questions": 10,
  "model": "meta-llama/llama-3.3-70b-instruct",
  "embedding_model": "intfloat/multilingual-e5-small",
  "top_k": 4,
  "retriever": {
    "recall_at_4": 1.0,
    "hits": 10
  },
  "generation": {
    "faithfulness": 0.8541666666666666,
    "answer_relevancy": 0.8618146549620485
  }
}
